> 📅 __Date: 2026-09-01__

# 🔎 **Retrievers in LangChain**

> **Goal:** Understand how LangChain retrievers find relevant documents for a user query, why a basic vector retriever is sometimes insufficient, and how specialized retrieval, reranking, metadata filtering, multi-vector retrieval, and contextual compression improve retrieval quality.

> **Prerequisite:** This chapter assumes familiarity with the previous RAG Architecture and RAG Implementation notes.

> **Dependencies:**

```python
!pip install -U langchain
!pip install -U langchain-classic
!pip install -U langchain-community
!pip install -U langchain-text-splitters
!pip install -U langchain-openai
!pip install -U langchain-chroma
!pip install -U chromadb
!pip install -U pypdf
!pip install -U rank_bm25
!pip install -U lark
```

**Install All Dependencies**
```python
!pip install -U langchain langchain-classic langchain-community langchain-text-splitters langchain-openai langchain-chroma chromadb pypdf rank_bm25 lark
```

---

# 🗺️ **Retriever Learning Roadmap**

```text
Basic Retriever
      ↓
Search Configuration
      ↓
 ┌───────────┬──────────────┬────────────────────────┐
 ↓           ↓              ↓
Similarity   MMR     Similarity Score Threshold
      ↓
Specialized Retrieval
      │
      ├── Multi-Query
      ├── Parent Document
      ├── Ensemble / Hybrid
      ├── Self-Query
      └── Multi-Vector
      ↓
Candidate Ranking / Filtering
      │
      ├── Reranker
      └── Metadata Filtering
      ↓
Contextual Compression
      │
      ├── LLMChainExtractor
      ├── LLMChainFilter
      └── EmbeddingsFilter
      ↓
Final Context
      ↓
RAG Answer
```

---


# 🎯 **What is a Retriever?**

> **Retriever = A component that takes a user query and returns relevant documents or chunks from a knowledge source.**

**In a vector-based RAG system:**

```text
User Query
    ↓
Query Embedding
    ↓
Vector Database
    ↓
Similarity Search
    ↓
Relevant Chunks
```

The retriever is responsible for **finding context**, not generating the final answer.

```text
Retriever
→ Find relevant information

LLM
→ Generate the answer
```

---

# 🧠 **Retriever in RAG**

```text
User Query
    ↓
Retriever
    ↓
Relevant Chunks
    ↓
Prompt + Context
    ↓
LLM
    ↓
Answer
```

> **A strong RAG system therefore needs both good retrieval and good generation.**

---

# 📄 **Example Knowledge Source**

**For the examples in this chapter:**

```text
NIPS-2017-attention-is-all-you-need-Paper.pdf
```

The document is loaded, split into chunks, embedded, and stored in a vector database before retrieval begins.

---

# 🏗️ **Basic RAG Setup**

## **1. Text Extraction**

In [1]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(
    "/content/NIPS-2017-attention-is-all-you-need-Paper.pdf"
)

docs = loader.load()

/tmp/ipykernel_34964/3330132712.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## **2. Chunking**

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

chunks = text_splitter.split_documents(docs)

## **3. Embeddings**

In [3]:
import os
from google.colab import userdata

openai = userdata.get("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = openai

In [4]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings()

## **4. Vector DB**

In [5]:
# from langchain_community.vectorstores import Chroma
from langchain_chroma import Chroma

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="db"
)

---

# 🔎 **Basic Retriever**

In [6]:
retriever = vector_db.as_retriever()

**Query the retriever:**

In [7]:
retriever.invoke(
    "What is positional encoding in transformer model?"
)

[Document(id='00fa3c8c-780f-4393-bea4-b9f331d4bed9', metadata={'source': '/content/NIPS-2017-attention-is-all-you-need-Paper.pdf', 'book': 'Advances in Neural Information Processing Systems 30', 'eventtype': 'Poster', 'creationdate': '', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'type': 'Conference Proceedings', 'published': '2017', 'creator': 'PyPDF', 'description': 'Paper accepted and presented at the Neural Information Processing Systems Conference (http://nips.cc/)', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality whi

**The conceptual flow is:**

```text
Query
 ↓
Embedding
 ↓
Vector DB
 ↓
Similarity Search
 ↓
Relevant Chunks
```

---

# 🧪 **Inspect Retrieved Chunks**

In [8]:
ret_chunks = retriever.invoke(
    "What is positional encoding in transformer model?"
)

for chunk in ret_chunks:
    print(chunk.page_content[:100])
    print("------------------------------------------------------------")

Similarly to other sequence transduction models, we use learned embeddings to convert the input
toke
------------------------------------------------------------
(E) positional embedding instead of sinusoids 4.92 25.7
big 6 1024 4096 16 0.3 300K 4.33 26.4 213
In
------------------------------------------------------------
position in the decoder to attend over all positions in the input sequence. This mimics the
typical 
------------------------------------------------------------
of the softmax which correspond to illegal connections. See Figure 2.
3.3 Position-wise Feed-Forward
------------------------------------------------------------


It is important to inspect retrieved content during RAG development.

> **Before blaming the LLM for a bad answer, check what the retriever actually returned.**

---

# 📊 **Choosing the Number of Chunks**

**The number of results can be configured with `k`:**

```python
retriever = vector_db.as_retriever(
    search_kwargs={"k": 6}
)
```

**Here:**

```text
k = 6
```

means the retriever requests up to six results from the underlying search operation.

> **`k` is configurable. It is not a universal RAG default.**

---

# 🧠 **What is `k`?**

> **`k` = Number of top retrieval results requested.**

**Example:**

```text
k = 4
→ Top 4 results

k = 6
→ Top 6 results
```

**More is not always better:**

```text
k ↑
 ↓
More Context
 ↓
Potentially More Cost / Noise
```

Choose `k` using retrieval evaluation and the available context budget.

---

# 📐 **Similarity Score Intuition**

**Suppose a search produces:**

```text
Chunk 1 → 0.92
Chunk 2 → 0.70
Chunk 3 → 0.20
Chunk 4 → 0.50
Chunk 5 → 0.90
```

A higher score generally indicates stronger relevance under the configured scoring system.

**Conceptually:**

```text
0.92 → Very High
0.90 → Very High
0.70 → Medium
0.50 → Lower
0.20 → Low
```

> **The exact meaning and scale of scores depend on the vector store and its relevance / distance configuration.**

---

# 🕵️ **Search Types**

**Three common search strategies are:**

```text
1. similarity
2. mmr
3. similarity_score_threshold
```

---

# 1️⃣ **Similarity Search**

> **Similarity search returns the documents most similar to the query according to the configured vector-search metric.**

In [9]:
retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 6}
)

# 🧪 **Inspect ```search_type="similarity"``` Retrieved Chunks**

In [10]:
similarity_chunks = retriever.invoke("what is position encoding in transformer model?")

for i in similarity_chunks:
    print(i.page_content[:100])
    print("------------------------------------------------------------")

Similarly to other sequence transduction models, we use learned embeddings to convert the input
toke
------------------------------------------------------------
(E) positional embedding instead of sinusoids 4.92 25.7
big 6 1024 4096 16 0.3 300K 4.33 26.4 213
In
------------------------------------------------------------
position in the decoder to attend over all positions in the input sequence. This mimics the
typical 
------------------------------------------------------------
of the softmax which correspond to illegal connections. See Figure 2.
3.3 Position-wise Feed-Forward
------------------------------------------------------------
Figure 1: The Transformer - model architecture.
wise fully connected feed-forward network. We employ
------------------------------------------------------------
aligned RNNs or convolution. In the following sections, we will describe the Transformer, motivate
s
------------------------------------------------------------


**Flow:**

```text
Query
 ↓
Query Vector
 ↓
Compare with Chunk Vectors
 ↓
Rank
 ↓
Top-k Chunks
```

**A common conceptual similarity measure is cosine similarity:**

$$
\operatorname{sim}(q,d)
=
\frac{q\cdot d}
{\|q\|\|d\|}
$$

**where:**

```text
q → Query vector
d → Document / chunk vector
```

---

# ✅ **When is Similarity Search Useful?**

```text
Specific semantic query
Simple retrieval requirement
Most similar chunks are usually enough
```

---

# 2️⃣ **MMR — Maximum Marginal Relevance**

> **MMR = Maximum Marginal Relevance**

**MMR tries to balance:**

```text
Relevance to Query
+
Diversity among Retrieved Results
```

Why?

**Suppose several top chunks contain nearly the same information:**

```text
Chunk 1 → Very relevant
Chunk 2 → Very relevant + almost identical
Chunk 3 → Very relevant + almost identical
Chunk 4 → Relevant + different information
```

Pure similarity may over-select redundant chunks.

MMR tries to reduce that redundancy.

---

# 🧠 **MMR Mental Model**

```text
Query
 ↓
Find relevant candidates
 ↓
Check similarity among candidates
 ↓
Penalize redundant results
 ↓
Select relevant + diverse results
```

**A common conceptual objective is:**

$$
\operatorname{MMR}(d)
=
\lambda\operatorname{sim}(d,q)
-
(1-\lambda)
\max_{d'\in S}\operatorname{sim}(d,d')
$$

**where:**

```text
d → Candidate document
q → Query
S → Already selected documents
λ → Relevance / diversity trade-off
```

---

# 🧪 **MMR Configuration**

In [11]:
retriever = vector_db.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 6}
)

# 🧪 **Inspect ```search_type="mmr"``` Retrieved Chunks**

In [12]:
mmr_chunks = retriever.invoke("what is position encoding in transformer model?")

for i in mmr_chunks:
    print(i.page_content[:100])
    print("------------------------------------------------------------")

Similarly to other sequence transduction models, we use learned embeddings to convert the input
toke
------------------------------------------------------------
(E) positional embedding instead of sinusoids 4.92 25.7
big 6 1024 4096 16 0.3 300K 4.33 26.4 213
In
------------------------------------------------------------
The goal of reducing sequential computation also forms the foundation of the Extended Neural GPU
[20
------------------------------------------------------------
PE (pos,2i) =sin(pos/100002i/dmodel)
PE (pos,2i+1) =cos(pos/100002i/dmodel)
wherepos is the position
------------------------------------------------------------
connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,
r
------------------------------------------------------------
single-precision ﬂoating-point capacity of each GPU 5.
6.2 Model Variations
To evaluate the importan
------------------------------------------------------------


**Flow:**

```text
Query
 ↓
Candidate Results
 ↓
Query Relevance
+
Result Diversity
 ↓
MMR Selection
 ↓
Relevant + Diverse Results
```

---

# ✅ **When is MMR Useful?**

```text
Results are repetitive
Query has multiple relevant aspects
You want broader information coverage
You want to reduce duplicate context
```

---

# 3️⃣ **Similarity Score Threshold**

> **Similarity-score threshold retrieval returns only results that satisfy a configured relevance condition.**

**Example:**

In [13]:
retriever = vector_db.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 6,
        "score_threshold": 0.8
    }
)

# 🧪 **Inspect ```search_type="similarity_score_threshold"``` Retrieved Chunks**

In [14]:
similarity_score_threshold_chunks = retriever.invoke("what is position encoding in transformer model?")

for i in similarity_score_threshold_chunks:
    print(i.page_content[:100])
    print("------------------------------------------------------------")

Similarly to other sequence transduction models, we use learned embeddings to convert the input
toke
------------------------------------------------------------


**Conceptually:**

```text
Query
 ↓
Similarity Scores
 ↓
Apply Threshold
 ↓
Keep sufficiently relevant results
```

**Example intuition:**

```text
Chunk 1 → 0.92 → ✅
Chunk 2 → 0.70 → ❌
Chunk 3 → 0.20 → ❌
Chunk 4 → 0.85 → ✅
```

> **Do not assume `0.8` means exactly the same thing across all vector stores. Calibrate thresholds on your own retrieval data.**

---

# 🆚 **Similarity vs MMR vs Threshold**

| Search Type | Main Goal | Key Idea |
|---|---|---|
| **Similarity** | Highest relevance | Select the most similar chunks |
| **MMR** | Relevance + diversity | Reduce redundant results |
| **Threshold** | Minimum relevance | Reject results below a configured condition |

**Memory trick:**

```text
Similarity
→ Most Similar

MMR
→ Similar + Diverse

Threshold
→ Relevant Enough
```

---

# 🧠 **Why Do We Need Specialized Retrievers?**

Basic semantic retrieval is powerful, **but real applications can have additional requirements:**

```text
Different query phrasing
Precise retrieval + richer context
Exact keywords / IDs
Metadata filters
Noise inside retrieved chunks
```

This leads to specialized strategies.

---

# 🔎 **Multi-Query Retriever**

## **Problem**

**A user can express the same intent in many ways:**

```text
"What is positional encoding?"

"Explain positional encoding"

"Tell me about positional encoding"

"Why is positional encoding used?"
```

A single query formulation may not retrieve everything useful.

---

# 💡 **Multi-Query Idea**

> **Multi-Query Retriever = Uses an LLM to generate multiple query variations and retrieves documents using those variations.**

**Basic retriever:**

```text
Query
 ↓
Embedding
 ↓
Vector DB
 ↓
Relevant Chunks
```

**Multi-Query:**

```text
Original Query
      ↓
     LLM
      ↓
 ┌────┼────┐
 ↓    ↓    ↓
Q1   Q2   Q3
 \    |    /
  \   |   /
   ↓  ↓  ↓
Retrieval
   ↓
Combine / Deduplicate
   ↓
Final Results
```

---

# 🧠 **Multi-Query Architecture**

```text
                     USER QUERY
                          ↓
                         LLM
                          ↓
          ┌───────────────┼───────────────┐
          ↓               ↓               ↓
       Query 1         Query 2         Query 3
          │               │               │
          └───────────────┼───────────────┘
                          ↓
                  Base Retrieval
                          ↓
                  Retrieved Chunks
                          ↓
                   Merge / Unique
                          ↓
                   Final Results
```

---

# 🧪 **Multi-Query Implementation**

In [15]:
from langchain_openai import OpenAI

llm = OpenAI()

base_retriever = vector_db.as_retriever()

In [16]:
from langchain_classic.retrievers import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm,
)

**Invoke:**

In [17]:
results = multi_query_retriever.invoke(
    "What is positional encoding in transformer model?"
)

**Inspect:**

In [18]:
for i, chunk in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print(chunk.page_content[:200])


--- Result 1 ---
Similarly to other sequence transduction models, we use learned embeddings to convert the input
tokens and output tokens to vectors of dimensiondmodel. We also use the usual learned linear transfor-
m

--- Result 2 ---
(E) positional embedding instead of sinusoids 4.92 25.7
big 6 1024 4096 16 0.3 300K 4.33 26.4 213
In Table 3 rows (B), we observe that reducing the attention key size dk hurts model quality. This
sugg

--- Result 3 ---
PE (pos,2i) =sin(pos/100002i/dmodel)
PE (pos,2i+1) =cos(pos/100002i/dmodel)
wherepos is the position andi is the dimension. That is, each dimension of the positional encoding
corresponds to a sinusoid

--- Result 4 ---
Figure 1: The Transformer - model architecture.
wise fully connected feed-forward network. We employ a residual connection [10] around each of
the two sub-layers, followed by layer normalization [ 1].

--- Result 5 ---
of the softmax which correspond to illegal connections. See Figure 2.
3.3 Position-wise Feed-Forward Netwo

---

# ✅ **Advantages of Multi-Query**

```text
Better query coverage
Handles different phrasings
Useful for ambiguous questions
Can retrieve information missed by one formulation
```

# ⚠️ **Limitations of Multi-Query**

```text
Extra LLM call
Higher latency
Higher cost
Generated queries may be poor
More retrieved context may increase downstream prompt size
```

**Conceptually:**

$$
\text{Query Coverage}\uparrow
\quad\Longrightarrow\quad
\text{Potential Retrieval Cost}\uparrow
$$

---

> 📅 __Date: 2026-09-02__

# 📄 **Parent Document Retriever**

## **The Chunk Size Trade-off**

**There is a common tension:**

```text
Small Chunk
→ Precise retrieval

Large Chunk
→ Better surrounding context
```

**Example:**

```text
4,000 characters
        ↓
1,000 + 1,000 + 1,000 + 1,000
```

Small chunks can match a query more precisely, but may not provide enough context to the LLM.

---

# 💡 **Parent-Child Strategy**

> **Parent Document Retriever = Search smaller child chunks for precision, but return their larger parent chunks for richer context.**

**Structure:**

```text
Parent
 ├── Child 1
 ├── Child 2
 ├── Child 3
 └── Child 4
```

**Retrieval flow:**

```text
Query
 ↓
Search Child Chunks
 ↓
Matching Child
 ↓
Find Parent
 ↓
Return Parent Context
```

---

# 🧩 **Parent Document Architecture**

```text
                 PARENT DOCUMENT
                       │
             ┌─────────┼─────────┐
             ↓         ↓         ↓
          Child 1   Child 2   Child 3
             │         │         │
             └─────────┼─────────┘
                       ↓
                  Vector Store
                       ↑
                     Query
                       ↓
                Child Retrieval
                       ↓
                 Parent Lookup
                       ↓
                 Richer Context
```

---

# 📏 **Parent and Child Splitters**

In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,
    chunk_overlap=0
)

child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=0
)

**Conceptually:**

```text
Parent
→ ~4000 characters

Child
→ ~1000 characters
```

---

# 🗃️ **Child Vector Store**

In [20]:
new_db = Chroma(
    collection_name="child_chunks",
    embedding_function=embedding_model,
)

The vector store holds the searchable child representations.

---

# 💾 **Parent Document Store**

In [21]:
from langchain_core.stores import InMemoryStore

store = InMemoryStore()

**Conceptually:**

```text
Child Chunks
→ Vector Store

Parent Chunks
→ Document Store
```

---

# 🔗 **Create ParentDocumentRetriever**

In [22]:
from langchain_classic.retrievers import ParentDocumentRetriever

parent_retriever = ParentDocumentRetriever(
    parent_splitter=parent_splitter,
    child_splitter=child_splitter,
    vectorstore=new_db,
    docstore=store
)

**Add the documents:**

In [23]:
parent_retriever.add_documents(docs)

**Query:**

In [24]:
results = parent_retriever.invoke(
    "What is positional encoding in transformer model?"
)

# 🧪 **Inspect ```ParentDocumentRetriever``` Retrieved Chunks**

In [25]:
for i, chunk in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print(chunk.page_content[:200])


--- Result 1 ---
Table 3: Variations on the Transformer architecture. Unlisted values are identical to those of the base
model. All metrics are on the English-to-German translation development set, newstest2013. Liste

--- Result 2 ---
MultiHead(Q,K,V ) = Concat(head 1,..., headh)W O
where headi = Attention(QW Q
i ,KW K
i ,VW V
i )
Where the projections are parameter matricesW Q
i ∈ Rdmodel×dk,W K
i ∈ Rdmodel×dk,W V
i ∈ Rdmodel×dv
a

--- Result 3 ---
Recurrent models typically factor computation along the symbol positions of the input and output
sequences. Aligning the positions to steps in computation time, they generate a sequence of hidden
stat


---

# 🔄 **Parent Document Flow**

```text
Original Document
       ↓
Parent Splitter
       ↓
Parent Chunks
       ↓
Child Splitter
       ↓
Child Chunks
  ↙            ↘
Vector DB    Parent Store
  ↑
Query
  ↓
Search Child
  ↓
Return Parent
```

---

# ✅ **Advantages of Parent Document Retriever**

```text
Precise search
Better surrounding context
Reduces small-chunk / large-context trade-off
Useful for long documents
```

# ⚠️ **Limitations**

```text
More complex indexing
Two storage layers
More bookkeeping
Returned parent may still be larger than needed
```

---

# 🆚 **Basic vs Parent Document Retriever**

| Feature | Basic Retriever | Parent Document Retriever |
|---|---|---|
| **Search Unit** | Chunk | Child chunk |
| **Returned Unit** | Same chunk | Parent chunk |
| **Context** | Smaller | Larger |
| **Architecture** | Simple | Two-level |
| **Storage** | Usually one store | Vector store + parent store |

**Memory trick:**

```text
Basic
→ Search Small → Return Small

Parent Document
→ Search Small → Return Big
```

---

# 🔀 **Hybrid / Ensemble Retriever**

## **Problem**

Vector search is semantic, but some queries depend heavily on exact words, IDs, codes or names.

**Example:**

```text
"Give me details of policy number 87"
```

For such queries, keyword-based retrieval can be valuable.

---

# 🔤 **BM25**

> **BM25 = A widely used keyword / lexical ranking method.**

**Conceptually:**

```text
Query
 ↓
Keyword Matching
 ↓
BM25
 ↓
Relevant Documents
```

**BM25 is useful for:**

```text
Exact Terms
Names
Identifiers
Codes
Keyword-heavy Queries
```

---

# 🧠 **Why Combine BM25 + Vector Search?**

**The two methods capture different signals:**

```text
BM25
→ Lexical / keyword relevance

Vector Search
→ Semantic relevance
```

**Therefore:**

```text
Query
   │
   ├────────→ BM25
   │             ↓
   │        Keyword Results
   │
   └────────→ Vector Retriever
                 ↓
            Semantic Results
                 │
                 ↓
              Ensemble
                 ↓
            Final Results
```

---

# 🧪 **BM25 Retriever**

In [26]:
from langchain_classic.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(
    documents=chunks,
)

**Vector retriever:**

In [27]:
vector_retriever = vector_db.as_retriever()

**Ensemble:**

In [28]:
from langchain_classic.retrievers import EnsembleRetriever

ensemble_retriever = EnsembleRetriever(
    retrievers=[
        bm25_retriever,
        vector_retriever
    ]
)

**Query:**

In [29]:
results = ensemble_retriever.invoke(
    "What is positional encoding in transformer model?"
)

**Inspect:**

In [30]:
for i, chunk in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print(chunk.page_content[:200])


--- Result 1 ---
(E) positional embedding instead of sinusoids 4.92 25.7
big 6 1024 4096 16 0.3 300K 4.33 26.4 213
In Table 3 rows (B), we observe that reducing the attention key size dk hurts model quality. This
sugg

--- Result 2 ---
Table 1: Maximum path lengths, per-layer complexity and minimum number of sequential operations
for different layer types.n is the sequence length,d is the representation dimension,k is the kernel
siz

--- Result 3 ---
Similarly to other sequence transduction models, we use learned embeddings to convert the input
tokens and output tokens to vectors of dimensiondmodel. We also use the usual learned linear transfor-
m

--- Result 4 ---
PE (pos,2i) =sin(pos/100002i/dmodel)
PE (pos,2i+1) =cos(pos/100002i/dmodel)
wherepos is the position andi is the dimension. That is, each dimension of the positional encoding
corresponds to a sinusoid

--- Result 5 ---
position in the decoder to attend over all positions in the input sequence. This mimics the
typical encode

---

# 🧠 **Ensemble Mental Model**

```text
                         QUERY
                           │
                ┌──────────┴──────────┐
                ↓                     ↓
              BM25              Vector Search
                ↓                     ↓
         Keyword Results        Semantic Results
                └──────────┬──────────┘
                           ↓
                        Ensemble
                           ↓
                     Final Ranking
```

---

# 📐 **Ensemble Intuition**

**A simplified weighted-combination idea is:**

$$
S(d)
=
\sum_{i=1}^{n} w_iS_i(d)
$$

**where:**

```text
S_i(d) → Score from retriever i
w_i    → Weight for retriever i
```

The exact ranking / fusion algorithm depends on the ensemble implementation.

---

# ✅ **Advantages of Ensemble Retrieval**

```text
Combines keyword + semantic signals
Strong for exact identifiers and names
Useful for mixed query types
Can improve retrieval coverage
```

# ⚠️ **Limitations**

```text
More complexity
More retrieval computation
Requires tuning and evaluation
Different retrievers may return overlapping results
```

---

# 🆚 **BM25 vs Vector vs Ensemble**

| Retriever | Main Signal | Strong At |
|---|---|---|
| **BM25** | Lexical / keyword | Exact terms, IDs, names |
| **Vector** | Semantic | Meaning, paraphrases |
| **Ensemble** | Combined | Mixed requirements |

**Memory trick:**

```text
BM25
→ Exact Words

Vector
→ Meaning

Ensemble
→ Words + Meaning
```

---

# 🧠 **Self-Query Retriever**

## **Problem**

**A document contains more than text:**

```text
Document
 ├── page_content
 ├── metadata
 └── embedding
```

A user can ask a question that contains both semantic meaning and metadata conditions.

**Example:**

```text
"Give me movies about dreams released in 2010."
```

**This contains:**

```text
Semantic condition:
→ dreams

Metadata condition:
→ year = 2010
```

---

# 💡 **Self-Query Idea**

> **Self-Query Retriever = Uses an LLM to translate natural language into a search query plus structured metadata filters.**

**Instead of:**

```text
Query
 ↓
Vector Search
```

**we have:**

```text
Natural Language Query
 ↓
LLM
 ↓
Search Query + Metadata Filter
 ↓
Vector Store
 ↓
Results
```

---

# 🧩 **Movie Dataset Example**

In [31]:
from langchain_classic.schema import Document

docs = [
    Document(
        page_content="A bunch of scientists bring back dinosaurs and mayhem breaks loose",
        metadata={
            "year": 1993,
            "rating": 7.7,
            "genre": "science fiction",
        },
    ),

    Document(
        page_content="Leo DiCaprio gets lost in a dream within a dream within a dream within a ...",
        metadata={
            "year": 2010,
            "director": "Christopher Nolan",
            "rating": 8.2,
        },
    ),

    Document(
        page_content="A psychologist / detective gets lost in a series of dreams within dreams within dreams and Inception reused the idea",
        metadata={
            "year": 2006,
            "director": "Satoshi Kon",
            "rating": 8.6,
        },
    ),

    Document(
        page_content="A bunch of normal-sized women are supremely wholesome and some men pine after them",
        metadata={
            "year": 2019,
            "director": "Greta Gerwig",
            "rating": 8.3,
        },
    ),

    Document(
        page_content="Toys come alive and have a blast doing so",
        metadata={
            "year": 1995,
            "genre": "animated",
        },
    ),

    Document(
        page_content="Three men walk into the Zone, three men walk out of the Zone",
        metadata={
            "year": 1979,
            "director": "Andrei Tarkovsky",
            "genre": "thriller",
            "rating": 9.9,
        },
    ),
]

---

# 🔍 **Inspect Metadata**

In [32]:
docs[0].metadata

{'year': 1993, 'rating': 7.7, 'genre': 'science fiction'}

**Conceptually:**

```text
Document
 ├── page_content
 └── metadata
       ├── year
       ├── rating
       ├── genre
       └── director
```

---

# 🗃️ **Create Movie Vector Database**

In [33]:
movie_db = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    persist_directory="movie_db"
)

---

# 🔎 **Basic Retriever Example**

In [34]:
retriever = movie_db.as_retriever(
    search_kwargs={"k": 1}
)

**Try:**

In [35]:
retriever.invoke(
    "Give me a movie about dinosaurs"
)

[Document(id='30d29aed-d345-48c3-99fa-1a0aec81452a', metadata={'year': 1993, 'rating': 7.7, 'genre': 'science fiction'}, page_content='A bunch of scientists bring back dinosaurs and mayhem breaks loose')]

**and:**

In [36]:
retriever.invoke(
    "Give me a movie about dreams released in 2010"
)

[Document(id='65f74853-a1a7-4d50-8d5d-1988376a5c8a', metadata={'rating': 8.6, 'year': 2006, 'director': 'Satoshi Kon'}, page_content='A psychologist / detective gets lost in a series of dreams within dreams within dreams and Inception reused the idea')]

The second query contains both semantic and metadata intent.

---

# 🧾 **Describe Metadata Fields**

In [37]:
from langchain_classic.chains.query_constructor.schema import AttributeInfo

metadata_info = [
    AttributeInfo(
        name="genre",
        description="The genre of the movie",
        type="string",
    ),
    AttributeInfo(
        name="year",
        description="The year the movie was released",
        type="integer",
    ),
    AttributeInfo(
        name="director",
        description="The name of the movie director",
        type="string",
    ),
    AttributeInfo(
        name="rating",
        description="A 1-10 rating for the movie",
        type="float",
    ),
]

These descriptions tell the query-construction system what the available metadata fields mean.

---

# 🤖 **Create Self-Query Retriever**


```python
!pip install databricks-langchain
```

**with this:**


At first, I tried installing `databricks-langchain` because
`SelfQueryRetriever.from_llm()` was raising an error related to
`DatabricksVectorSearch`.

```python
!pip install databricks-langchain
```

However, the installation introduced additional dependency conflicts in the Colab environment, and DatabricksVectorSearch was still not the actual vector store being used in this example.

#### ⚠️ **Problem**

**When creating the Self-Query Retriever:**

```python
self_query_retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=movie_db,
    metadata_field_info=metadata_info,
    document_contents="Brief description of the movie"
)
```

**I received:**

```python
ImportError: cannot import name 'DatabricksVectorSearch'
from 'langchain_community.vectorstores'
```

The problem was caused by LangChain trying to automatically determine the vector-store translator and importing an incompatible ```DatabricksVectorSearch``` integration.

#### ✅ **Solution**

Since this example uses **Chroma**, explicitly provide the Chroma
structured-query translator instead of relying on automatic translator
discovery.

```python
from langchain_classic.retrievers.self_query.chroma import ChromaTranslator

self_query_retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=movie_db,
    metadata_field_info=metadata_info,
    document_contents="Brief description of the movie",
    structured_query_translator=ChromaTranslator()
)
```

> **Key Point:** We do not need databricks-langchain for this
Chroma-based Self-Query example. The error was caused by the automatic
translator lookup, not because Databricks Vector Search was required.

In [38]:
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_classic.retrievers.self_query.chroma import ChromaTranslator

self_query_retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=movie_db,
    metadata_field_info=metadata_info,
    document_contents="Brief description of the movie",
    structured_query_translator=ChromaTranslator()
)

In [39]:
results = self_query_retriever.invoke(
    "Give me movies about dreams released in 2010"
)

for doc in results:
    print(doc.page_content)
    print(doc.metadata)
    print("-" * 80)

Leo DiCaprio gets lost in a dream within a dream within a dream within a ...
{'director': 'Christopher Nolan', 'rating': 8.2, 'year': 2010}
--------------------------------------------------------------------------------


---

# 🔄 **Self-Query Flow**

**For:**

```text
"Give me a movie about dreams released in 2010."
```

**the conceptual process is:**

```text
Natural Language Query
        ↓
       LLM
        ↓
Semantic Search Query
+
Metadata Filter:
year = 2010
        ↓
Vector Store
        ↓
Filtered / Ranked Results
```

---

# ✅ **Advantages of Self-Query Retriever**

```text
Understands natural-language filters
Uses metadata during retrieval
Useful for structured document collections
Can express conditions such as year, rating, category, etc.
```

# ⚠️ **Limitations**

```text
Requires an LLM call
Metadata schema must be described correctly
Generated filters can be wrong
Vector-store filter support matters
More complex than basic retrieval
```

---

# 🆚 **Basic vs Self-Query**

| Feature | Basic Retriever | Self-Query Retriever |
|---|---|---|
| Semantic search | ✅ | ✅ |
| Natural-language metadata filtering | Manual / limited | ✅ |
| Query interpretation | Direct | LLM-assisted |
| Complexity | Low | Higher |
| LLM query-construction call | No | Yes |

Memory trick:

```text
Basic
→ Search Content

Self-Query
→ Search Content + Metadata
```

---

# 🗜️ **Contextual Compression**

## **Problem**

A retrieved chunk can be relevant but still contain a lot of irrelevant information.

**Example:**

```text
Retrieved Chunk
──────────────────────────
Relevant sentence
Unrelated sentence
Unrelated sentence
Relevant sentence
Additional noise
──────────────────────────
```

**Passing the whole chunk to the LLM may increase:**

```text
Prompt Size
Cost
Noise
Distraction
```

---

# 💡 **Contextual Compression Idea**

> **Contextual Compression = Retrieve relevant documents first, then compress those documents so only query-relevant information is kept.**

**Flow:**

```text
Query
 ↓
Retriever
 ↓
Retrieved Chunks
 ↓
Compressor
 ↓
Relevant Content Only
```

---

# 🧩 **Contextual Compression Architecture**

```text
                USER QUERY
                     ↓
                  Retriever
                     ↓
               Retrieved Chunks
                     ↓
                 Compressor
                     ↓
          Compressed Relevant Content
                     ↓
                    LLM
                     ↓
                  Answer
```

---

# 🔄 **Two-Step Process**

### **1. Retrieval**

```text
Query
 ↓
Base Retriever
 ↓
Relevant Chunks
```

### **2. Compression**

```text
Relevant Chunks
 ↓
Compressor
 ↓
Relevant Extracted Content
```

**Therefore:**

```text
Base Retriever
+
Compressor
=
Contextual Compression Retriever
```

---

# 🧪 **Inspect Base Retriever Results**

In [40]:
retriever = vector_db.as_retriever()

ret_chunks = retriever.invoke(
    "What is positional encoding in transformer model?"
)

for chunk in ret_chunks:
    print(chunk.page_content)
    print("------------------------------------------------------------")

Similarly to other sequence transduction models, we use learned embeddings to convert the input
tokens and output tokens to vectors of dimensiondmodel. We also use the usual learned linear transfor-
mation and softmax function to convert the decoder output to predicted next-token probabilities. In
our model, we share the same weight matrix between the two embedding layers and the pre-softmax
linear transformation, similar to [24]. In the embedding layers, we multiply those weights by√dmodel.
3.5 Positional Encoding
Since our model contains no recurrence and no convolution, in order for the model to make use of the
order of the sequence, we must inject some information about the relative or absolute position of the
tokens in the sequence. To this end, we add "positional encodings" to the input embeddings at the
5
------------------------------------------------------------
(E) positional embedding instead of sinusoids 4.92 25.7
big 6 1024 4096 16 0.3 300K 4.33 26.4 213
In Table 3 rows (

---

# 🤖 **LLMChainExtractor**

> **LLMChainExtractor = An LLM-based compressor that extracts the parts of retrieved documents that are relevant to the current query.**

**Create the compressor:**

In [41]:
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

compressor = LLMChainExtractor.from_llm(
    llm=llm
)

**Conceptually:**

```text
Retrieved Chunk
      +
User Query
      ↓
     LLM
      ↓
Relevant Extracted Content
```

---

# 🔗 **ContextualCompressionRetriever**

In [42]:
from langchain_classic.retrievers import ContextualCompressionRetriever

contextual_compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever
)

**Invoke it:**

In [43]:
ret_chunks = contextual_compression_retriever.invoke(
    "What is positional encoding in transformer model?"
)

**Inspect:**

In [44]:
for chunk in ret_chunks:
    print(chunk.page_content)
    print("------------------------------------------------------------")

- "convert the input tokens and output tokens to vectors of dimensiondmodel"
- "we share the same weight matrix between the two embedding layers and the pre-softmax linear transformation"
- "we multiply those weights by√dmodel"
- "Positional Encoding"
- "order of the sequence"
- "inject some information about the relative or absolute position of the tokens in the sequence"
- "positional encodings"
------------------------------------------------------------
positional embedding instead of sinusoids 4.92 25.7
big 6 1024 4096 16 0.3 300K 4.33 26.4 213
------------------------------------------------------------
- The encoder contains self-attention layers.
- Each position in the encoder can attend to all positions in the previous layer of the encoder.
- Similarly, self-attention layers in the decoder allow each position in the decoder to attend to all positions in the decoder up to and including that position.
- We need to prevent leftward information ﬂow in the decoder to preserve the a

# 🗜️ **LLMChainFilter**

> **LLMChainFilter = Uses an LLM to decide whether a retrieved document / chunk is relevant to the query. Irrelevant chunks are filtered out completely.**

This is slightly different from `LLMChainExtractor`.

### **LLMChainExtractor**

```text
Retrieved Chunk
      ↓
LLM
      ↓
Extract Relevant Information
      ↓
Smaller / Compressed Chunk
```

### **LLMChainFilter**

```text
Retrieved Chunk
      ↓
LLM
      ↓
Relevant?
   ┌──┴──┐
  YES    NO
   ↓      ↓
 Keep    Drop
```

**So:**

```text
LLMChainExtractor
→ Keep only relevant parts of a chunk

LLMChainFilter
→ Keep or drop the entire chunk
```

---

# 🧩 **LLMChainFilter Architecture**

```text
User Query
     ↓
Base Retriever
     ↓
Retrieved Chunks
     ↓
LLMChainFilter
     ↓
┌───────────────┐
│ Relevant?     │
└───────┬───────┘
        ↓
 ┌──────┴──────┐
 ↓             ↓
Keep           Drop
 ↓
Filtered Chunks
```

---

# 🧪 **LLMChainFilter Implementation**

In [45]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainFilter

retriever = vector_db.as_retriever()

compressor = LLMChainFilter.from_llm(
    llm=llm
)

contextual_compression_filter = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever
)

**Now invoke:**

In [46]:
ret_chunks = contextual_compression_filter.invoke(
    "What is positional encoding in transformer model?"
)

for chunk in ret_chunks:
    print(chunk.page_content)
    print("------------------------------------------------------------")

Similarly to other sequence transduction models, we use learned embeddings to convert the input
tokens and output tokens to vectors of dimensiondmodel. We also use the usual learned linear transfor-
mation and softmax function to convert the decoder output to predicted next-token probabilities. In
our model, we share the same weight matrix between the two embedding layers and the pre-softmax
linear transformation, similar to [24]. In the embedding layers, we multiply those weights by√dmodel.
3.5 Positional Encoding
Since our model contains no recurrence and no convolution, in order for the model to make use of the
order of the sequence, we must inject some information about the relative or absolute position of the
tokens in the sequence. To this end, we add "positional encodings" to the input embeddings at the
5
------------------------------------------------------------
(E) positional embedding instead of sinusoids 4.92 25.7
big 6 1024 4096 16 0.3 300K 4.33 26.4 213
In Table 3 rows (

---

# 🆚 **LLMChainExtractor vs LLMChainFilter**

| Compressor            | What it does                                         |
| --------------------- | ---------------------------------------------------- |
| **LLMChainExtractor** | Extracts only the relevant part of a retrieved chunk |
| **LLMChainFilter**    | Decides whether to keep or discard the entire chunk  |

**Memory trick:**

```text
Extractor
→ Extract part

Filter
→ Keep / Drop
```

---

# 🧠 **When to Use LLMChainFilter**

**Use it when:**

```text
Retrieved chunks may be only partially relevant
You want to remove completely irrelevant chunks
You want to keep an entire chunk when it is relevant
```

**A useful mental model:**

> **Extractor modifies the content; Filter modifies the set of documents.**

---

# 🧮 **Contextual Compression — Three Levels**

```text
Base Retriever
      ↓
Retrieved Chunks
      ↓
 ┌───────────────┬─────────────────┐
 ↓               ↓                 ↓
Extractor       Filter          Embeddings
 ↓               ↓                 ↓
Compress        Keep/Drop        Similarity
Content         Documents        Filter
```

---

# 🧲 **Embedding Filter**

> **EmbeddingsFilter = Uses embedding similarity to filter retrieved documents, keeping documents whose similarity meets the configured threshold.**

**The important difference is:**

```text
LLMChainFilter
→ Uses an LLM

EmbeddingsFilter
→ Uses embeddings
```

---

# 🆚 **EmbeddingsFilter vs Similarity Score Threshold Retriever**

These two ideas are closely related.

### **Similarity Score Threshold**

```text
Query
 ↓
Vector Search
 ↓
Similarity Score
 ↓
Threshold
 ↓
Keep / Drop
```

### **EmbeddingsFilter**

```text
Query
 ↓
Embedding
 ↓
Retrieved Documents
 ↓
Embedding Similarity
 ↓
Threshold
 ↓
Keep / Drop
```

The embedding filter can be used **after another retriever has already produced candidate documents**.

That makes it a flexible compression stage.

> **EmbeddingsFilter can be used as a post-retrieval filter rather than replacing the initial retrieval strategy.**

---

# 🧩 **EmbeddingsFilter Architecture**

```text
User Query
     ↓
Base Retriever
     ↓
Candidate Chunks
     ↓
Embedding Similarity
     ↓
Similarity Threshold
     ↓
Filtered Chunks
```

---

# 🧪 **EmbeddingsFilter Implementation**

In [47]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import EmbeddingsFilter

retriever = vector_db.as_retriever(
    search_type="mmr"
)

compressor = EmbeddingsFilter(
    embeddings=embedding_model,
    similarity_threshold=0.85
)

contextual_compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever
)

**Now invoke:**

In [48]:
ret_chunks = contextual_compression_retriever.invoke(
    "What is positional encoding in transformer model?"
)

for chunk in ret_chunks:
    print(chunk.page_content)
    print("------------------------------------------------------------")

Similarly to other sequence transduction models, we use learned embeddings to convert the input
tokens and output tokens to vectors of dimensiondmodel. We also use the usual learned linear transfor-
mation and softmax function to convert the decoder output to predicted next-token probabilities. In
our model, we share the same weight matrix between the two embedding layers and the pre-softmax
linear transformation, similar to [24]. In the embedding layers, we multiply those weights by√dmodel.
3.5 Positional Encoding
Since our model contains no recurrence and no convolution, in order for the model to make use of the
order of the sequence, we must inject some information about the relative or absolute position of the
tokens in the sequence. To this end, we add "positional encodings" to the input embeddings at the
5
------------------------------------------------------------


---

# 🔄 **Why Use MMR Before EmbeddingsFilter?**

**The pipeline can be:**

```text
Query
 ↓
MMR Retriever
 ↓
Relevant + Diverse Candidate Chunks
 ↓
EmbeddingsFilter
 ↓
Remove Low-Similarity Candidates
 ↓
Final Context
```

**This combines:**

```text
MMR
→ Relevance + Diversity

EmbeddingsFilter
→ Similarity Threshold
```

**So the overall result can be:**

```text
Relevant
+
Diverse
+
Above Similarity Threshold
```

---

# 🆚 **LLMChainFilter vs EmbeddingsFilter**

| Feature                    | LLMChainFilter                | EmbeddingsFilter              |
| -------------------------- | ----------------------------- | ----------------------------- |
| **Technique**              | LLM-based                     | Embedding-based               |
| **Decision**               | LLM judges relevance          | Similarity score              |
| **Extra LLM call**         | ✅                             | ❌                             |
| **Speed**                  | Generally slower              | Generally faster              |
| **Cost**                   | Higher                        | Lower                         |
| **Semantic understanding** | Stronger / more flexible      | Based on embedding similarity |
| **Threshold**              | Not primarily threshold-based | ✅                             |
| **Entire chunk**           | Keep / Drop                   | Keep / Drop                   |

**Memory trick:**

```text
LLMChainFilter
→ Ask the LLM: "Is this relevant?"

EmbeddingsFilter
→ Ask the vectors: "Is this similar enough?"
```

---

# 🧠 **Contextual Compression — Complete View**

**Contextual compression means:**

```text
Base Retriever
      ↓
Retrieved Documents
      ↓
Compression / Filtering
      ↓
Better Context
      ↓
LLM
```

**There can be different compressor strategies:**

```text
LLMChainExtractor
→ Extract relevant content

LLMChainFilter
→ Keep / Drop documents

EmbeddingsFilter
→ Similarity-based Keep / Drop
```

---

# 🏆 **Reranker**

> **Reranker = A second-stage ranking model that takes retrieved candidates and reorders them according to their relevance to the query.**

**A common retrieval architecture is:**

```text
Query
 ↓
Initial Retriever
 ↓
Top-N Candidate Documents
 ↓
Reranker
 ↓
Top-K Best Documents
 ↓
LLM
```

---

# 🧠 **Why Do We Need a Reranker?**

The first-stage retriever is optimized for fast candidate retrieval.

**It may retrieve:**

```text
Top 20 Candidates
```

but not rank every candidate perfectly.

A reranker performs a more detailed relevance evaluation:

```text
20 Candidates
      ↓
Reranker
      ↓
Best 5
```

**This is often called:**

```text
Two-Stage Retrieval
```

---

# 🔄 **Two-Stage Retrieval**

```text
                 QUERY
                   ↓
             First-Stage Retriever
                   ↓
             Candidate Documents
                   ↓
                Reranker
                   ↓
            Best Relevant Documents
                   ↓
                  LLM
```

**The first stage focuses on:**

```text
Recall
→ Find enough potentially relevant candidates
```

**The second stage focuses more on:**

```text
Precision
→ Rank the best candidates higher
```

> **A common pattern is: retrieve broadly first, then rerank more precisely.**

---

# 🆚 **Retriever vs Reranker**

| Component     | Main Responsibility                    |
| ------------- | -------------------------------------- |
| **Retriever** | Find candidate documents quickly       |
| **Reranker**  | Reorder candidates by deeper relevance |

**Memory trick:**

```text
Retriever
→ Find candidates

Reranker
→ Rank candidates
```

---

# 🏷️ **Metadata Filtering**

**A document can contain:**

```text
page_content
+
metadata
+
embedding
```

Sometimes the user query includes a condition that should be applied directly to metadata.

**Example:**

```text
"Show documents from 2025 about machine learning."
```

**Here:**

```text
Semantic condition
→ machine learning

Metadata condition
→ year = 2025
```

---

# 🧩 **Metadata Filtering Flow**

```text
User Query
     ↓
Semantic Search
     +
Metadata Filter
     ↓
Filtered Candidate Set
     ↓
Relevant Results
```

**Conceptually:**

```text
Query
 ↓
┌──────────────────────┐
│ Semantic Retrieval   │
└──────────┬───────────┘
           +
┌──────────────────────┐
│ Metadata Filter      │
│ year = 2025          │
└──────────┬───────────┘
           ↓
      Final Results
```

---

# 🧠 **Why Metadata Filtering Matters**

**Consider a knowledge base containing:**

```text
Company Policy
Product Documentation
Research Papers
News Articles
```

**Each document may have metadata such as:**

```text
source
department
year
author
document_type
```

**Instead of searching everything:**

```text
Entire Vector DB
```

**we can restrict the search:**

```text
department = "HR"
```

**or:**

```text
year = 2025
```

This can improve both relevance and efficiency.

---

# 🆚 **Semantic Search vs Metadata Filtering**

| Type                   | Matches Based On      |
| ---------------------- | --------------------- |
| **Semantic Search**    | Meaning of the text   |
| **Metadata Filtering** | Structured attributes |

**A powerful retrieval system can combine both:**

```text
Semantic Meaning
+
Structured Metadata
```

---

# 🧩 **Multi-Vector Retriever**

> **Multi-Vector Retriever = A retrieval strategy where multiple vector representations can be associated with a single parent document or source item.**

**Instead of representing one document with only one vector:**

```text
Document
 ↓
One Vector
```

**we may represent it using multiple vectors:**

```text
Document
 ├── Vector 1
 ├── Vector 2
 ├── Vector 3
 └── ...
```

---

# 🤔 **Why Multiple Vectors?**

A single vector may not represent every important aspect of a document equally well.

**Multiple vectors can represent:**

```text
Different passages
Different summaries
Different questions
Different views of the same document
```

This can improve retrieval coverage for complex knowledge bases.

---

# 🧩 **Multi-Vector Architecture**

```text
                     DOCUMENT
                         │
          ┌──────────────┼──────────────┐
          ↓              ↓              ↓
      Representation  Representation  Representation
          1              2              3
          ↓              ↓              ↓
       Vector 1       Vector 2       Vector 3
          └──────────────┼──────────────┘
                         ↓
                    Vector Store
                         ↑
                         │
                       Query
                         ↓
                    Matching Vectors
                         ↓
                 Retrieve Original Doc
```

---

# 🧠 **Multi-Vector Mental Model**

```text
One Document
      ↓
Many Representations
      ↓
Many Vectors
      ↓
Better Ways to Match Queries
      ↓
Original Document
```

---

# 🔗 **Relationship with Parent Document Retrieval**

Parent Document Retrieval and Multi-Vector Retrieval are related ideas but are not identical.

### **Parent Document Retriever**

```text
Parent
 ↓
Child Chunks
 ↓
Child Vectors
 ↓
Retrieve Parent
```

### **Multi-Vector Retriever**

```text
Document
 ↓
Multiple Representations
 ↓
Multiple Vectors
 ↓
Retrieve Associated Document
```

**The key difference:**

```text
Parent Document
→ Focuses on child-to-parent context

Multi-Vector
→ Focuses on multiple vector representations for retrieval
```

---

# 🏗️ **Advanced RAG Retrieval Architecture**

```text
                         USER QUERY
                              │
                              ↓
                    QUERY PROCESSING
                              │
             ┌────────────────┼────────────────┐
             ↓                ↓                ↓
         Multi-Query       Self-Query      Normal Query
             │                │                │
             └────────────────┼────────────────┘
                              ↓
                      INITIAL RETRIEVAL
                    ┌─────────┼─────────┐
                    ↓                   ↓
                 BM25              Vector Search
                    └─────────┬─────────┘
                              ↓
                           Ensemble
                              ↓
                        Candidate Set
                              ↓
                           Reranker
                              ↓
                     Contextual Compression
                              │
                 ┌────────────┼────────────┐
                 ↓            ↓            ↓
             Extractor      Filter     EmbeddingsFilter
                 └────────────┼────────────┘
                              ↓
                       Final Context
                              ↓
                             LLM
                              ↓
                           ANSWER
```

---

# 🧠 **How to Choose the Retrieval Strategy**

```text
Basic semantic search works?
            │
           YES
            ↓
      Keep Similarity
            │
           NO
            ↓
   What is the problem?
            │
     ┌──────┼──────────────┬─────────────┐
     ↓      ↓              ↓             ↓
  Query   Context       Keywords      Metadata
  wording  size          matter        matters
     ↓      ↓              ↓             ↓
 Multi-   Parent        Ensemble      Self-Query
 Query    Document
            │
            ↓
       Results noisy?
            ↓
      Contextual
      Compression
            │
            ↓
      Ranking still weak?
            ↓
         Reranker
```

---

# 🧩 **Retriever Selection Guide**

```text
                   Start
                     │
                     ↓
             Basic semantic search
                     │
                Is it enough?
                ┌────┴────┐
               YES        NO
                ↓          ↓
             Use it     Diagnose
                           │
      ┌──────────┬─────────┼──────────┬─────────────┐
      ↓          ↓         ↓          ↓             ↓
  Different   Need      Need       Metadata       Noise
  phrasing   larger    keyword     filters      in chunks
             context   matching
      ↓          ↓         ↓          ↓             ↓
 Multi-Query  Parent   Ensemble   Self-Query   Compression
             Document
```

---

# 🆚 **Retriever Comparison**

| Retriever / Technique | Main Problem Solved | Core Strategy | Typical Role |
|---|---|---|---|
| **Similarity** | Find semantically related chunks | Vector similarity | Basic retrieval |
| **MMR** | Reduce redundant results | Relevance + diversity | Basic retrieval |
| **Threshold** | Reject weak matches | Minimum relevance condition | Basic retrieval |
| **Multi-Query** | Different query phrasings | LLM-generated query variants | Query expansion |
| **Parent Document** | Precision vs context trade-off | Search child, return parent | Context retrieval |
| **BM25** | Exact words / identifiers | Lexical matching | Keyword retrieval |
| **Ensemble / Hybrid** | Semantic + lexical gap | Combine retrievers | Candidate generation |
| **Self-Query** | Natural-language metadata conditions | LLM query + metadata filter | Query-aware retrieval |
| **Multi-Vector** | One source needs multiple representations | Multiple vectors → same source | Candidate generation |
| **Reranker** | Candidate ordering | Second-stage relevance scoring | Ranking |
| **Metadata Filtering** | Restrict search space | Structured field filters | Retrieval constraint |
| **Contextual Compression** | Noise inside retrieved chunks | Retrieve → compress/filter | Post-retrieval |

> **These techniques can be combined. They are not mutually exclusive, and the best pipeline depends on the query, documents, metadata, quality target, latency budget, and cost.**

---

# 🧠 **Ultimate Memory Tricks**

### **Similarity**

```text
→ Most Similar
```

### **MMR**

```text
→ Similar + Diverse
```

### **Threshold**

```text
→ Relevant Enough
```

### **Multi-Query**

```text
→ One Question → Many Query Versions
```

### **Parent Document**

```text
→ Search Small → Return Big
```

### **Ensemble**

```text
→ Keywords + Meaning
```

### **Self-Query**

```text
→ Query + Metadata Filter
```

### **Contextual Compression**

```text
→ Retrieve + Remove Noise
```

---

# 🧠 **Important Retrieval Principle**

There is no universally best retriever.

**The right choice depends on:**

```text
Query Type
+
Document Structure
+
Metadata
+
Chunking Strategy
+
Information Density
+
Latency
+
Cost
```

**Conceptually:**

$$
\text{Best Retriever}
=
f(
\text{Query},
\text{Documents},
\text{Metadata},
\text{Latency},
\text{Cost}
)
$$

---

# 🔬 **Retriever Debugging Workflow**

**When the RAG answer is poor:**

```text
User Query
     ↓
Check Query
     ↓
Retriever
     ↓
Inspect Retrieved Documents
     ↓
Is the required information present?
     │
   ┌─┴─┐
  YES  NO
   ↓    ↓
Check   Improve
Prompt  Retrieval
   ↓     │
  LLM    ├── Chunking
         ├── Search Type
         ├── k
         ├── Multi-Query
         ├── Ensemble
         ├── Self-Query
         └── Compression
```

> **If the correct information was never retrieved, changing the prompt alone cannot reliably fix the retrieval failure.**

---

# 🧪 **Retriever Inspection Helper**

In [49]:
def inspect_retriever(retriever, query, preview_chars=200):
    results = retriever.invoke(query)

    print(f"Query: {query}")
    print(f"Retrieved documents: {len(results)}")

    for i, doc in enumerate(results, start=1):
        print(f"\n--- Result {i} ---")
        print(doc.page_content[:preview_chars])
        print("Metadata:", doc.metadata)

**Use:**

In [50]:
inspect_retriever(
    retriever,
    "What is positional encoding in transformer model?"
)

Query: What is positional encoding in transformer model?
Retrieved documents: 4

--- Result 1 ---
Similarly to other sequence transduction models, we use learned embeddings to convert the input
tokens and output tokens to vectors of dimensiondmodel. We also use the usual learned linear transfor-
m
Metadata: {'creationdate': '', 'moddate': '2018-02-12T21:22:10-08:00', 'page_label': '5', 'creator': 'PyPDF', 'page': 4, 'subject': 'Neural Information Processing Systems http://nips.cc/', 'created': '2017', 'source': '/content/NIPS-2017-attention-is-all-you-need-Paper.pdf', 'eventtype': 'Poster', 'date': '2017', 'language': 'en-US', 'published': '2017', 'producer': 'PyPDF2', 'book': 'Advances in Neural Information Processing Systems 30', 'firstpage': '5998', 'publisher': 'Curran Associates, Inc.', 'title': 'Attention is All you Need', 'lastpage': '6008', 'type': 'Conference Proceedings', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvo

---

# 📊 **Retriever Evaluation**

Evaluate retrieval independently from generation.

**Useful questions:**

```text
Did we retrieve the correct chunk?
Did we retrieve enough context?
Did we retrieve too much irrelevant context?
Are the results diverse?
Are metadata filters correct?
Are exact IDs / names retrieved?
```

**Common retrieval metrics include:**

```text
Recall
Precision
Hit Rate
Top-k Accuracy
MRR
NDCG
```

The exact metric should match the application's goal.

---

# 🏗️ **Retriever in the Complete RAG System**

```text
             KNOWLEDGE BASE
                    ↓
                CHUNKING
                    ↓
                EMBEDDING
                    ↓
                VECTOR DB
                    ↓
                 RETRIEVER
                    ↓
        ┌───────────┼────────────┐
        ↓           ↓            ↓
      Basic      Specialized  Compression
        ↓           ↓            ↓
 Similarity /  Multi-Query /   Remove
 MMR /         Parent /       Noise
 Threshold     Ensemble /
               Self-Query
        └───────────┼────────────┘
                    ↓
             RELEVANT CONTEXT
                    ↓
                AUGMENTATION
                    ↓
               QUERY + CONTEXT
                    ↓
                   LLM
                    ↓
                 ANSWER
```

---

# 🧠 **Complete Retriever Mental Model**

```text
                         USER QUERY
                              │
                              ↓
                     QUERY UNDERSTANDING
                              │
          ┌───────────────────┼───────────────────┐
          ↓                   ↓                   ↓
     Multi-Query          Self-Query          Normal Query
          │                   │                   │
          │             Metadata Filter            │
          └───────────────────┼───────────────────┘
                              ↓
                     INITIAL RETRIEVAL
                ┌─────────────┼─────────────┐
                ↓             ↓             ↓
            Similarity       MMR      Ensemble / BM25
                │             │             │
                └─────────────┼─────────────┘
                              ↓
                       Candidate Set
                              ↓
                    Optional Reranker
                              ↓
                 Contextual Compression
                    ┌─────────┼─────────┐
                    ↓         ↓         ↓
                Extractor   Filter   EmbeddingsFilter
                    └─────────┼─────────┘
                              ↓
                        Final Context
                              ↓
                       QUERY + CONTEXT
                              ↓
                             LLM
                              ↓
                           ANSWER
```

> **The retrieval layer is not just one vector search. It can be a sequence of query expansion, candidate generation, filtering, ranking, and compression steps designed to deliver the most useful context to the LLM.**

---

# 🆚 **Quick Selection Cheat Sheet**

| Requirement | Good Starting Point |
|---|---|
| Simple semantic matching | **Similarity** |
| Repetitive results | **MMR** |
| Reject weak matches | **Similarity Score Threshold** |
| Different query phrasings | **Multi-Query** |
| Precise search + larger context | **Parent Document** |
| Exact terms + semantic meaning | **Ensemble / Hybrid** |
| Natural-language metadata conditions | **Self-Query** |
| Multiple representations for one source | **Multi-Vector** |
| Candidate ordering is weak | **Reranker** |
| Search space should be restricted by fields | **Metadata Filtering** |
| Retrieved chunks contain noise | **Contextual Compression** |

---

# 🎓 **Interview-Friendly Explanation**

> **If an interviewer asks: "What is a retriever in RAG?"**

```text
A retriever is the component responsible for finding relevant
chunks or documents from the indexed knowledge base for a user query.

In a basic vector retriever, the query is converted into an embedding
and compared with stored document embeddings. The most relevant chunks
are returned and then passed to the LLM as context.

When basic retrieval is not enough, we can use MMR to improve diversity,
Multi-Query to generate alternate query formulations, Parent Document
Retriever to search smaller child chunks but return larger parent context,
Ensemble Retriever to combine keyword and semantic retrieval, Self-Query
for natural-language metadata conditions, Multi-Vector retrieval when one
source needs multiple vector representations, a reranker to improve candidate
ordering, Metadata Filtering to restrict the search space, and Contextual
Compression when the retrieved chunks contain unnecessary information.
```

---

# 🧠 **Interview: Similarity vs MMR**

```text
Similarity mainly prioritizes relevance to the query.

MMR also considers redundancy among the selected documents.

So:
Similarity → Relevance
MMR        → Relevance + Diversity
```

---

# 🧠 **Interview: Multi-Query vs Self-Query**

```text
Multi-Query
→ Generates multiple query formulations.

Self-Query
→ Interprets the user's natural language and can construct
  a search query plus metadata filters.
```

**Memory trick:**

```text
Multi-Query
→ Multiple Questions

Self-Query
→ Structured Search
```

---

# 🧠 **Interview: Parent Document vs Basic Retriever**

```text
A basic retriever searches and returns the same chunk.

A Parent Document Retriever searches smaller child chunks for precision,
but returns their larger parent chunks for richer context.
```

---

# 🧠 **Interview: BM25 vs Vector Search**

```text
BM25 is mainly lexical, so it is strong for exact terms,
names, identifiers and keywords.

Vector search is semantic, so it is strong for meaning and paraphrases.

An ensemble retriever can combine both signals.
```

---

# 🧠 **Interview: What is Contextual Compression?**

```text
Contextual compression first retrieves relevant documents and then
compresses those documents so only the information relevant to the
current query is passed downstream.
```

---

# ⚠️ **Common Beginner Mistakes**

## **1. Thinking a Retriever Generates the Answer**

```text
Retriever
→ Finds context

LLM
→ Generates answer
```

---

## **2. Thinking More `k` is Always Better**

```text
k ↑
 ↓
More Context
 ↓
More Tokens / Potential Noise
```

---

## **3. Ignoring Retrieval Quality**

```text
Wrong Retrieval
      ↓
Wrong Context
      ↓
LLM
      ↓
Potentially Wrong Answer
```

---

## **4. Treating Every Query as Pure Semantic Search**

**Some applications need:**

```text
Exact Keywords
Metadata Filters
Multiple Query Formulations
Diverse Results
Larger Context
Noise Reduction
```

---

## **5. Using Multi-Query for Every Query**

**Multi-Query adds an LLM step:**

```text
Better Coverage
+
Extra Cost / Latency
```

Use it only when the retrieval benefit is worth the overhead.

---

## **6. Compressing Too Aggressively**

```text
Compression
→ Less Noise

But potentially:
→ Less Useful Information
```

Evaluate the compressed context before using it in production.

---

# ⚠️ **Legacy API Note**

**Several examples in this chapter use classic-style imports such as:**

```python
from langchain_classic.retrievers import ...
```

These examples are useful for learning the retrieval concepts and for working with corresponding classic APIs.

> **LangChain APIs and package organization evolve over time. For a new project, check the current official documentation for the recommended import paths and retriever APIs.**

---

# 📋 **Quick Revision**

```text
Retriever
→ Finds relevant documents / chunks

k
→ Number of requested top results

Similarity
→ Most similar chunks

MMR
→ Relevance + diversity

Similarity Score Threshold
→ Reject weak matches

Multi-Query Retriever
→ One query → multiple query variants

Parent Document Retriever
→ Search child → return parent

BM25
→ Keyword / lexical retrieval

Ensemble / Hybrid Retriever
→ Combine lexical + semantic signals

Self-Query Retriever
→ Query meaning + metadata filters

Multi-Vector Retriever
→ One source → multiple vector representations

Reranker
→ Reorder retrieved candidates

Metadata Filtering
→ Restrict retrieval using structured fields

Contextual Compression
→ Retrieve → reduce irrelevant context

LLMChainExtractor
→ Extract relevant parts of retrieved documents

LLMChainFilter
→ Keep / drop retrieved documents using an LLM

EmbeddingsFilter
→ Keep / drop retrieved documents using embedding similarity
```

---

# 🧠 **One-Line Retrieval Formula**

```text
Retriever
=
Query
→ Search
→ Relevant Documents
```

**Specialized retrieval:**

```text
Multi-Query
=
One Query
→ Many Query Versions
→ Retrieve
```

```text
Parent Document
=
Search Small
→ Return Large
```

```text
Ensemble
=
Keyword
+
Semantic
```

```text
Self-Query
=
Query Meaning
+
Metadata Filter
```

```text
Multi-Vector
=
One Source
→ Multiple Vectors
```

```text
Reranker
=
Retrieve Candidates
→ Reorder by Relevance
```

```text
Compression
=
Retrieve
+
Remove Irrelevant Context
```

---

# 🏁 **Key Takeaways**

> **1. A retriever is responsible for finding relevant documents or chunks for a user query.**

> **2. A basic vector retriever uses embeddings and vector search to find semantically related chunks.**

> **3. `k` controls how many top results are requested from the search.**

> **4. Similarity prioritizes relevance, while MMR balances relevance with diversity.**

> **5. Similarity thresholds can reject weak matches, but threshold values must be calibrated to the retrieval system.**

> **6. Multi-Query improves coverage by generating alternative query formulations.**

> **7. Parent Document Retriever searches child chunks but returns parent context.**

> **8. BM25 is lexical retrieval, vector search is semantic retrieval, and Ensemble combines retrieval signals.**

> **9. Self-Query can translate natural language into a search query plus structured metadata filters.**

> **10. Multi-Vector retrieval allows multiple vector representations to point to the same underlying source.**

> **11. Rerankers perform a second-stage ordering of retrieved candidates.**

> **12. Metadata Filtering restricts retrieval using structured document attributes.**

> **13. Contextual Compression reduces unnecessary information after retrieval.**

> **14. `LLMChainExtractor`, `LLMChainFilter`, and `EmbeddingsFilter` are different compression/filtering strategies.**

> **15. There is no universally best retriever; the correct choice depends on the data and query requirements.**

> **16. Retrieval should be evaluated separately from generation because retrieval failures can be the root cause of incorrect RAG answers.**

---

# 🎯 **Final Mental Model**

```text
                         USER QUERY
                              │
                              ↓
                  Optional Query Processing
                  ┌───────────┼───────────┐
                  ↓           ↓           ↓
             Multi-Query   Self-Query   Normal Query
                  │           │
                  │      Metadata Filter
                  └───────────┼───────────┘
                              ↓
                     Candidate Generation
          ┌───────────────────┼───────────────────┐
          ↓                   ↓                   ↓
     Similarity / MMR      BM25 / Hybrid      Multi-Vector
          └───────────────────┼───────────────────┘
                              ↓
                       Candidate Set
                              ↓
                          Reranker
                              ↓
                    Contextual Compression
                   ┌──────────┼──────────┐
                   ↓          ↓          ↓
               Extractor   Filter   EmbeddingsFilter
                   └──────────┼──────────┘
                              ↓
                         Final Context
                              ↓
                       QUERY + CONTEXT
                              ↓
                             LLM
                              ↓
                           ANSWER
```

> **A good RAG retriever does not simply return more documents. It uses the appropriate retrieval, filtering, ranking, and compression steps to return the right information with the right amount of context.**

---

# 🔗 **Useful Resource**

> **Retriever APIs and package organization can change between LangChain releases. Check the current official documentation when implementing these patterns in a new project.**

```text
LangChain Documentation
https://docs.langchain.com
```

---

# 🏁 **End Note**

```text
Basic Retrieval
→ Try Similarity

Need Diversity
→ MMR

Need Minimum Relevance
→ Threshold

Different Query Phrasing
→ Multi-Query

Need More Context After Precise Matching
→ Parent Document

Need Keyword + Semantic Search
→ Ensemble / Hybrid

Need Metadata Filtering
→ Self-Query

Need Less Noise
→ Contextual Compression
```

> **Retriever selection is an engineering decision: match the retrieval strategy to the structure of the knowledge, the type of user query, and the application's latency, cost, and quality requirements.**